<!-- repro-header -->
> **Reproducibility header**
>
> - **Aim:** Download of public datasets from GEO.
> - **Manuscript panels:** Data collection
> - **Inputs:** GEO accessions
> - **Outputs:** Raw data folders
> - **Environment / kernel:** `geoparse`
> - Data root: `/lustre/groups/ml01/datasets/projects/2025_sergio_troutpy` (ROIs on Zenodo 10.5281/zenodo.17446348). 


In [20]:
!pip3 install requests beautifulsoup4 tqdm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


In [26]:
import os
import urllib.request
from bs4 import BeautifulSoup
from tqdm import tqdm
import requests

# List of GSM IDs
gsm_ids = ["GSE234713"]

# Output directory
output_dir = "/media/sergio/Meninges/troutpy/raw/cosmx_colon"
os.makedirs(output_dir, exist_ok=True)

# Base URL for GSM pages
gsm_base_url = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc="


def download_ftp_file(url, out_path):
    try:
        urllib.request.urlretrieve(url, out_path)
    except Exception as e:
        print(f"Failed to download {url}: {e}")


for gsm_id in gsm_ids:
    print(f"Processing {gsm_id}")
    url = gsm_base_url + gsm_id
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    # Look for FTP links
    ftp_links = [a["href"] for a in soup.find_all("a", href=True) if a["href"].startswith("ftp://")]

    # Create GSM-specific folder
    gsm_folder = os.path.join(output_dir, gsm_id)
    os.makedirs(gsm_folder, exist_ok=True)

    for link in tqdm(ftp_links, desc=f"Downloading files for {gsm_id}"):
        filename = os.path.basename(link)
        out_path = os.path.join(gsm_folder, filename)
        if not os.path.exists(out_path):
            download_ftp_file(link, out_path)

Processing GSE234713


In [27]:
from urllib.parse import unquote

filename = unquote(os.path.basename(link))

In [28]:
filename

'GSE234713_ReadMe_SMI_Data_File.html'

In [1]:
import GEOparse

In [2]:
# Replace with your GEO accession
gse_id = "GSE234713"

# Download the data to a folder (it won't re-download if cached)
gse = GEOparse.get_GEO(geo=gse_id, destdir="/media/sergio/Meninges/troutpy/raw/cosmx_colon")

# Inspect what's inside
print(gse)

10-Apr-2025 12:06:25 DEBUG utils - Directory /media/sergio/Meninges/troutpy/raw/cosmx_colon already exists. Skipping.
10-Apr-2025 12:06:25 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE234nnn/GSE234713/soft/GSE234713_family.soft.gz to /media/sergio/Meninges/troutpy/raw/cosmx_colon/GSE234713_family.soft.gz
100%|██████████████████████████████████████████████████████████████████████████████| 5.01k/5.01k [00:00<00:00, 8.88kB/s]
10-Apr-2025 12:06:26 DEBUG downloader - Size validation passed
10-Apr-2025 12:06:26 DEBUG downloader - Moving /tmp/tmpmqgdzki_ to /media/sergio/Meninges/troutpy/raw/cosmx_colon/GSE234713_family.soft.gz
10-Apr-2025 12:06:26 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE234nnn/GSE234713/soft/GSE234713_family.soft.gz
10-Apr-2025 12:06:26 INFO GEOparse - Parsing /media/sergio/Meninges/troutpy/raw/cosmx_colon/GSE234713_family.soft.gz: 
10-Apr-2025 12:06:26 DEBUG GEOparse - DATABASE: GeoMiame
10-Apr-2025 12:06:2

<SERIES: GSE234713 - 9 SAMPLES, 1 d(s)>


In [3]:
gse

<SERIES: GSE234713 - 9 SAMPLES, 1 d(s)>

In [4]:
# Sample (GSM) metadata
for gsm_name, gsm in gse.gsms.items():
    print(f"{gsm_name}: {gsm.metadata['title']}")

# Check for expression data
for gsm in gse.gsms.values():
    print(gsm.table.head())
    break  # just show one

GSM7473682: ['HC a_non-IBD colon']
GSM7473683: ['HC b_non-IBD colon']
GSM7473684: ['HC c_non-IBD colon']
GSM7473685: ['UC a_UC colon']
GSM7473686: ['UC b_UC colon']
GSM7473687: ['UC c_UC colon']
GSM7473688: ['CD a_CD colon']
GSM7473689: ['CD b_CD colon']
GSM7473690: ['CD c_CD colon']
Empty DataFrame
Columns: []
Index: []


In [5]:
import os

output_dir = "./geo_data/processed_tables"
os.makedirs(output_dir, exist_ok=True)

for gsm_name, gsm in gse.gsms.items():
    out_path = os.path.join(output_dir, f"{gsm_name}.csv")
    gsm.table.to_csv(out_path)

In [8]:
gsm

AttributeError: 'GSM' object has no attribute 'keys'

In [6]:
import GEOparse
import os

# Replace with your GEO Series ID
gse_id = "GSE234713"  # example: "GSE2034"

# Download the dataset
gse = GEOparse.get_GEO(geo=gse_id, destdir="/media/sergio/Meninges/troutpy/raw/cosmx_colon", how="full")

# Create output directory
output_dir = f"./geo_data/{gse_id}_samples"
os.makedirs(output_dir, exist_ok=True)

# Iterate over all GSMs (samples)
for gsm_name, gsm in gse.gsms.items():
    print(f"Saving sample: {gsm_name}")

    # Save expression table (if available)
    if gsm.table is not None and not gsm.table.empty:
        gsm.table.to_csv(f"{output_dir}/{gsm_name}.csv", index=False)
    else:
        print(f"No expression table found for {gsm_name}")

10-Apr-2025 12:08:35 DEBUG utils - Directory /media/sergio/Meninges/troutpy/raw/cosmx_colon already exists. Skipping.
10-Apr-2025 12:08:35 INFO GEOparse - File already exist: using local version.
10-Apr-2025 12:08:35 INFO GEOparse - Parsing /media/sergio/Meninges/troutpy/raw/cosmx_colon/GSE234713_family.soft.gz: 
10-Apr-2025 12:08:35 DEBUG GEOparse - DATABASE: GeoMiame
10-Apr-2025 12:08:35 DEBUG GEOparse - SERIES: GSE234713
10-Apr-2025 12:08:35 DEBUG GEOparse - PLATFORM: GPL33484
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473682
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473683
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473684
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473685
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473686
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473687
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473688
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473689
10-Apr-2025 12:08:35 DEBUG GEOparse - SAMPLE: GSM7473690


Saving sample: GSM7473682
No expression table found for GSM7473682
Saving sample: GSM7473683
No expression table found for GSM7473683
Saving sample: GSM7473684
No expression table found for GSM7473684
Saving sample: GSM7473685
No expression table found for GSM7473685
Saving sample: GSM7473686
No expression table found for GSM7473686
Saving sample: GSM7473687
No expression table found for GSM7473687
Saving sample: GSM7473688
No expression table found for GSM7473688
Saving sample: GSM7473689
No expression table found for GSM7473689
Saving sample: GSM7473690
No expression table found for GSM7473690


In [18]:
print(gsm)

<SAMPLE: GSM7473690>


In [12]:
print(gsm.platform)  # GEO accession (e.g., GPL96)
print(gsm.platform.table)  # Feature annotation (e.g., gene mapping info)

AttributeError: 'GSM' object has no attribute 'platform'

In [10]:
print(gsm.metadata.keys())
# Example output:
# dict_keys(['title', 'source_name_ch1', 'organism_ch1', 'characteristics_ch1', 'sample_type', 'description', ...])

AttributeError: 'GSM' object has no attribute 'data'